# 🌊 PINNs-MVP: Kolmogorov Flow 實驗指南
## Physics-Informed Neural Networks with Leith Turbulence Prior for 2D Flow

---

**最後更新**: 2025-12-18  
**版本**: v4.3 (Leith Prior Edition)  
**新功能**: ✅ Leith Turbulence Prior | ✅ Leith QR Sensor | ✅ Momentum Merging | ✅ Exponential LR Decay

---

### 📚 快速導航

1. **環境設定** - Colab 初始化與 GPU 檢查
2. **Leith 數據準備** - 檢查低保真場與感測點
3. **模型訓練** - 使用 Leith Prior 訓練 PINNs
4. **結果評估** - 視覺化與物理驗證

---

### 🎯 核心特色

- **Leith 湍流先驗**: 使用 Leith 渦黏模型場作為軟約束（更適合 2D 湍流）
- **智能感測器**: 從 Leith 場生成 QR-Pivot 最優感測點
- **Momentum Merging**: 針對各向同性流動簡化損失項（3→2，-33%）
- **Exponential LR Decay**: 平滑學習率衰減策略（gamma=0.95, step=500 epochs）
- **完整物理**: Fourier Features + SIREN + 自適應權重 + 因果訓練
- **生產級訓練**: 10000 epochs 完整訓練（需 GPU 長時間運行）

---

### ⚙️ Google Colab 推薦配置

| GPU 類型 | 訓練時間 (10000 epochs) | 訂閱 | 建議 |
|---------|------------------------|------|------|
| **NVIDIA A100** | 8-12 小時 ⭐ | Colab Pro+ | 最佳選擇 |
| **NVIDIA V100** | 12-18 小時 | Colab Pro | 可接受 |
| **NVIDIA T4** | 20-30 小時 ⚠️ | 免費/Pro | 需分段訓練 |

**⚠️ 重要提示**: 
- **免費 Colab 不建議**：T4 會因超時中斷（12小時限制）
- **建議使用 Colab Pro+**：A100 可在單次 session 完成
- **替代方案**：使用 checkpoint resume 機制分段訓練

---

## Part 0: Google Colab 初始化

**⚠️ 重要**：本 Notebook 專為 **Google Colab (T4/A100 GPU)** 設計。

In [ ]:
# 0.1 檢測環境
try:
    import google.colab
    IN_COLAB = True
    print("✅ Google Colab 環境")
except ImportError:
    IN_COLAB = False
    print("⚠️  本地環境，請使用: python scripts/train/train.py --cfg <config.yml>")

In [ ]:
# 0.2 掛載 Google Drive 並設定模組路徑
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    import sys
    PROJECT_PATH = '/content/drive/MyDrive/pinns-mvp'

    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"✅ 專案目錄: {os.getcwd()}")

        # 將專案根目錄加入 Python 模組搜尋路徑
        if PROJECT_PATH not in sys.path:
            sys.path.insert(0, PROJECT_PATH)
            print(f"✅ 已加入模組路徑: {PROJECT_PATH}")

        # 驗證 pinnx 模組可以導入
        try:
            import pinnx
            print(f"✅ pinnx 模組載入成功")
        except ImportError as e:
            print(f"❌ 無法導入 pinnx: {e}")
            print(f"   請確認專案已完整上傳至 Google Drive")
    else:
        raise FileNotFoundError(f"專案不存在: {PROJECT_PATH}\n請上傳專案至 Google Drive")
else:
    import os
    import sys
    project_root = os.getcwd()
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        print(f"✅ 本地環境，已加入模組路徑: {project_root}")

In [ ]:
# 0.3 檢查 GPU
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"記憶體: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("⚠️ 僅 CPU，訓練將很慢")
    device = 'cpu'

---

## Part 1: Leith 數據準備

### 1.1 檢查 Leith 低保真場

In [ ]:
# 1.1 檢查 Leith 數據
import h5py
import numpy as np

leith_file = 'data/lowfi/kolmogorov_rans/rans_re50_kf4_leith.h5'

with h5py.File(leith_file, 'r') as f:
    print("📊 Leith 數據結構:")
    print(f"  群組: {list(f.keys())}")
    print(f"\n  mean_field 包含: {list(f['mean_field'].keys())}")

    u = f['mean_field']['u'][:]
    v = f['mean_field']['v'][:]
    nu_t = f['mean_field']['nu_t'][:]  # 渦黏度
    x = f['mean_field']['x'][:]  # 1D 座標
    y = f['mean_field']['y'][:]  # 1D 座標

    print(f"\n  網格大小: {u.shape}")
    print(f"  座標格式: x{x.shape}, y{y.shape} (1D arrays)")
    print(f"  u 範圍: [{u.min():.4f}, {u.max():.4f}]")
    print(f"  v 範圍: [{v.min():.4f}, {v.max():.4f}]")
    print(f"  nu_t 範圍: [{nu_t.min():.4e}, {nu_t.max():.4e}]")
    print(f"\n  ✅ Leith 模型特點: 僅包含 u, v, nu_t (無 k/epsilon)")

### 1.2 視覺化 Leith 場

In [ ]:
# 1.2 繪製 Leith 場
import matplotlib.pyplot as plt

with h5py.File(leith_file, 'r') as f:
    u = f['mean_field']['u'][:]
    v = f['mean_field']['v'][:]
    x = f['mean_field']['x'][:]
    y = f['mean_field']['y'][:]
    nu_t = f['mean_field']['nu_t'][:]

# 生成 2D meshgrid（從 1D 座標）
X, Y = np.meshgrid(x, y)

speed = np.sqrt(u**2 + v**2)
vorticity = np.gradient(v, axis=1) - np.gradient(u, axis=0)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# 速度場
im1 = ax1.contourf(X, Y, speed, levels=50, cmap='viridis')
ax1.set_title('Leith Speed Field', fontsize=14, fontweight='bold')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
plt.colorbar(im1, ax=ax1, label='|u|')

# 渦量場
im2 = ax2.contourf(X, Y, vorticity, levels=50, cmap='RdBu_r')
ax2.set_title('Leith Vorticity', fontsize=14, fontweight='bold')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
plt.colorbar(im2, ax=ax2, label='ω')

# 渦黏度場
im3 = ax3.contourf(X, Y, nu_t, levels=50, cmap='plasma')
ax3.set_title('Leith Eddy Viscosity', fontsize=14, fontweight='bold')
ax3.set_xlabel('x')
ax3.set_ylabel('y')
plt.colorbar(im3, ax=ax3, label='νₜ')

plt.tight_layout()
plt.savefig('results/leith_field_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Leith 場視覺化完成")

### 1.3 檢查 Leith QR Sensor

In [ ]:
# 1.3 檢查 Leith sensor
sensor_file = 'data/lowfi/kolmogorov_rans/sensors_K100_leith.npz'

sensors = np.load(sensor_file)

print("📍 Leith Sensor 配置:")
print(f"  感測點數: {sensors['K']}")
print(f"  方法: {sensors['method']}")
print(f"  來源: {sensors['source']} (Leith turbulence model)")
print(f"  X 範圍: [{sensors['sensor_x'].min():.4f}, {sensors['sensor_x'].max():.4f}]")
print(f"  Y 範圍: [{sensors['sensor_y'].min():.4f}, {sensors['sensor_y'].max():.4f}]")

metrics = sensors['metrics'].item()
print(f"\n  條件數: {metrics.get('condition_number', 'N/A'):.2e}")

if metrics.get('condition_number', 1000) < 100:
    print("  ✅ 優秀（< 100）")
elif metrics.get('condition_number', 1000) < 500:
    print("  ✅ 良好（100-500）")
else:
    print("  ⚠️ 可接受（> 500）")

### 1.4 視覺化感測點分佈

In [ ]:
# 1.4 繪製感測點疊加在 Leith 場上
fig, ax = plt.subplots(figsize=(10, 10))

# Leith 背景
with h5py.File(leith_file, 'r') as f:
    u = f['mean_field']['u'][:]
    v = f['mean_field']['v'][:]
    x = f['mean_field']['x'][:]
    y = f['mean_field']['y'][:]

X, Y = np.meshgrid(x, y)
speed = np.sqrt(u**2 + v**2)
im = ax.contourf(X, Y, speed, levels=50, cmap='viridis', alpha=0.8)

# 感測點
ax.scatter(sensors['sensor_x'], sensors['sensor_y'],
           c='red', s=30, marker='x', linewidths=2,
           label=f"QR Sensors (K={sensors['K']})")

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Leith QR-Pivot Sensors Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_aspect('equal')
plt.colorbar(im, ax=ax, label='Speed |u|')

plt.tight_layout()
plt.savefig('results/leith_sensor_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Leith Sensor 視覺化完成")

---

## Part 2: 模型訓練

### 2.1 檢查配置文件

In [ ]:
# 2.1 檢查配置文件
import yaml

config_file = 'configs/kolmogorov_re50_kf4_K100.yml'

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

print("📋 訓練配置摘要:")
print(f"  實驗名稱: {config['experiment']['name']}")
print(f"  版本: {config['experiment']['version']}")
print(f"\n🔧 模型:")
print(f"  類型: {config['model']['type']}")
print(f"  寬度: {config['model']['width']}")
print(f"  深度: {config['model']['depth']}")
print(f"  Fourier: {config['model']['fourier_features']['enabled']}")
print(f"\n🎯 訓練:")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  優化器: {config['training']['optimizer']['type']}")
print(f"  初始學習率: {config['training']['optimizer']['lr']}")
print(f"\n📉 學習率調度:")
scheduler = config['training']['lr_scheduler']
print(f"  類型: {scheduler['type']}")
print(f"  Gamma: {scheduler.get('gamma', 'N/A')}")
print(f"  Step Size: {scheduler.get('step_size', 'N/A')} epochs")
print(f"  說明: LR 每 {scheduler.get('step_size', 'N/A')} epochs 衰減至 {scheduler.get('gamma', 'N/A')} 倍")
print(f"\n🌊 Leith Prior:")
print(f"  啟用: {config['lowfi_prior']['enabled']}")
print(f"  初始權重: {config['lowfi_prior']['consistency_weight']}")
print(f"  數據: {config['lowfi_prior']['data_path']}")
print(f"\n📚 Curriculum Learning:")
if config.get('curriculum', {}).get('enable', False):
    print(f"  啟用: True")
    stages = config['curriculum']['stages']
    print(f"  階段數: {len(stages)}")
    for i, stage in enumerate(stages, 1):
        epoch_range = stage['epoch_range']
        prior_w = stage['lowfi_prior']['consistency_weight']
        cont_w = stage['weights']['continuity']
        print(f"    Stage {i} ({epoch_range[0]}-{epoch_range[1]}): Prior={prior_w}, Continuity={cont_w}")
else:
    print(f"  啟用: False")
print(f"\n⚡ 損失配置:")
print(f"  Momentum Merging: {config['losses'].get('merge_momentum', False)}")
print(f"  說明: {'各向同性優化（X/Y 同步）' if config['losses'].get('merge_momentum') else '標準模式（X/Y 分離）'}")
print(f"\n📍 感測器:")
print(f"  K: {config['sensors']['K']}")
print(f"  方法: {config['sensors']['selection_method']}")

### 2.2 啟動訓練（GPU 加速）

**⚠️ 訓練時間警告**：
- 本配置為**生產級完整訓練**（10000 epochs）
- **不建議在免費 Colab 執行**（會超時中斷）
- **建議環境**：Colab Pro+ (A100) 或本地 GPU
- **替代方案**：修改配置文件中的 `epochs` 為 1000-2000 進行快速驗證

In [ ]:
# 2.2 訓練模型（使用 Leith Prior + Exponential LR Decay）
# 訓練時間：T4 ~20-30 小時 | V100 ~12-18 小時 | A100 ~8-12 小時（10000 epochs）

import os

# 設定 PYTHONPATH 讓子進程能找到 pinnx 模組
if IN_COLAB:
    os.environ['PYTHONPATH'] = '/content/drive/MyDrive/pinns-mvp'
else:
    os.environ['PYTHONPATH'] = os.getcwd()

print(f"✅ PYTHONPATH: {os.environ['PYTHONPATH']}")

# 執行訓練
!python scripts/train/train.py --cfg configs/kolmogorov_re50_kf4_K100.yml --device {device}

# 檢查點保存至: checkpoints/kolmogorov_re50_kf4_K100_leith_prior/
# 結果保存至: results/kolmogorov_re50_kf4_K100_leith_prior/

print("\n" + "="*70)
print("📊 訓練完成提示")
print("="*70)
print("如需與 vanilla baseline 對比，可執行：")
print("  !python scripts/train/train.py --cfg configs/kolmogorov_re50_kf4_K100_vanilla.yml --device {device}")
print("\n兩個版本使用相同的 exponential scheduler，差異僅在 Leith prior 和 curriculum。")
print("="*70)

### 2.3 監控訓練進度

In [ ]:
# 2.3 檢查訓練日誌
import os

log_dir = 'log'
log_files = [f for f in os.listdir(log_dir) if 'kolmogorov_re50' in f]

if log_files:
    latest_log = sorted(log_files)[-1]
    print(f"📄 最新日誌: {latest_log}\n")
    !tail -30 log/{latest_log}
else:
    print("⚠️ 未找到訓練日誌")

In [ ]:
# 2.4 列出檢查點
checkpoint_dir = 'checkpoints/kolmogorov_re50_kf4_K100_leith_prior'

if os.path.exists(checkpoint_dir):
    checkpoints = sorted(os.listdir(checkpoint_dir))
    print(f"📂 檢查點 ({len(checkpoints)} 個):\n")
    for ckpt in checkpoints[-5:]:  # 顯示最近 5 個
        path = os.path.join(checkpoint_dir, ckpt)
        size = os.path.getsize(path) / 1e6
        print(f"  {ckpt} ({size:.1f} MB)")
else:
    print("⚠️ 檢查點目錄不存在")

---

## Part 3: 結果評估

### 3.1 評估最佳模型

In [ ]:
# 3.1 使用統一評估腳本
!python scripts/evaluate/evaluate_checkpoint.py \
  --checkpoint checkpoints/kolmogorov_re50_kf4_K100_leith_prior/best_model.pth \
  --config configs/kolmogorov_re50_kf4_K100.yml \
  --output results/evaluation_leith_prior/

# 生成指標：
# - 相對 L2 誤差 (u, v, p)
# - 物理殘差 (連續性、動量)
# - 壓力梯度誤差 (dpdx, dpdy)

print("\n" + "="*70)
print("💡 提示：如需評估 vanilla baseline，執行：")
print("="*70)
print("!python scripts/evaluate/evaluate_checkpoint.py \\")
print("  --checkpoint checkpoints/kolmogorov_re50_kf4_K100_vanilla/best_model.pth \\")
print("  --config configs/kolmogorov_re50_kf4_K100_vanilla.yml \\")
print("  --output results/evaluation_vanilla/")
print("="*70)

### 3.2 視覺化結果

In [ ]:
# 3.2 生成完整視覺化
!python scripts/visualize/visualize_results.py \
  --checkpoint checkpoints/kolmogorov_re50_kf4_K100_leith_prior/best_model.pth \
  --reference data/kolmogorov_dns/dns_re50_t100.h5 \
  --output results/evaluation_leith_prior/visualizations/

# 生成圖表：
# - field_comparison.png: 預測 vs 真值 vs 誤差
# - energy_spectrum.png: 能譜對比
# - statistics.png: 統計量分析

print("\n" + "="*70)
print("📊 對比視覺化建議")
print("="*70)
print("如需生成 Leith prior vs Vanilla 的對比圖表，可以：")
print("1. 先分別生成兩個版本的視覺化結果")
print("2. 使用自訂腳本合併兩組結果進行並排比較")
print("3. 或者使用 evaluate_checkpoint.py 的 metrics.json 進行量化對比")
print("="*70)

In [ ]:
# 3.3 顯示結果圖
from IPython.display import Image, display

viz_dir = 'results/evaluation_leith_prior/visualizations'

if os.path.exists(viz_dir):
    print("📊 場重建對比:")
    display(Image(filename=f'{viz_dir}/field_comparison.png', width=1200))

    print("\n📈 能譜對比:")
    display(Image(filename=f'{viz_dir}/energy_spectrum.png', width=800))
else:
    print("⚠️ 視覺化結果尚未生成")

### 3.3 量化評估總結

In [ ]:
# 3.4 讀取評估指標
import json

metrics_file = 'results/evaluation_leith_prior/metrics.json'

if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)

    print("="*70)
    print("🏆 評估總結")
    print("="*70)

    print("\n📊 場重建誤差:")
    u_l2 = metrics['field_errors']['u_l2_error'] * 100
    v_l2 = metrics['field_errors']['v_l2_error'] * 100
    p_l2 = metrics['field_errors']['p_l2_error'] * 100

    print(f"  u: {u_l2:.2f}% {'✅' if u_l2 < 15 else '❌'} (目標 < 15%)")
    print(f"  v: {v_l2:.2f}% {'✅' if v_l2 < 15 else '❌'} (目標 < 15%)")
    print(f"  p: {p_l2:.2f}% {'✅' if p_l2 < 20 else '❌'} (目標 < 20%)")

    if 'pressure_gradient' in metrics:
        print("\n📐 壓力梯度誤差 (Leith Prior 改善):")
        dpdx = metrics['pressure_gradient']['dpdx_l2'] * 100
        dpdy = metrics['pressure_gradient']['dpdy_l2'] * 100
        print(f"  ∂p/∂x: {dpdx:.2f}% {'✅' if dpdx < 30 else '❌'} (目標 < 30%)")
        print(f"  ∂p/∂y: {dpdy:.2f}% {'✅' if dpdy < 30 else '❌'} (目標 < 30%)")

    print("\n⚖️ 物理守恆:")
    div = metrics['physics']['divergence_error']
    print(f"  連續性: {div:.2e} {'✅' if div < 1e-3 else '❌'} (目標 < 1e-3)")

    print("\n" + "="*70)
else:
    print("⚠️ 評估指標文件不存在")

---

## 📚 參考資料

### 📖 文檔
- [`docs/TECHNICAL_DOCUMENTATION.md`](docs/TECHNICAL_DOCUMENTATION.md) - 完整技術文檔
- [`docs/MOMENTUM_MERGING_GUIDE.md`](docs/MOMENTUM_MERGING_GUIDE.md) - Momentum Merging 使用指南
- [`REFACTORING_COMPLETE_GUIDE.md`](REFACTORING_COMPLETE_GUIDE.md) - 代碼重構指南
- [`configs/templates/README.md`](configs/templates/README.md) - 配置模板指南
- [`scripts/README.md`](scripts/README.md) - 腳本使用說明

### 🛠️ 關鍵腳本
- `scripts/train/train.py` - 主訓練器
- `scripts/evaluate/evaluate_checkpoint.py` - 檢查點評估
- `scripts/visualize/visualize_results.py` - 結果視覺化
- `scripts/generate/sensors/generate_sensors_periodic_qr.py` - QR-Pivot 感測器

### 📊 配置文件
**本 Notebook 使用的配置**：
- `configs/kolmogorov_re50_kf4_K100.yml` - **Leith Prior 版本**（10k epochs）
  - ✅ Leith 先驗引導（Curriculum: 10.0 → 1.0 → 0.1）
  - ✅ Exponential LR Decay（gamma=0.95, step=500）
  - ✅ 3-Stage Curriculum Learning
  
- `configs/kolmogorov_re50_kf4_K100_vanilla.yml` - **Vanilla Baseline**（10k epochs）
  - ❌ 無 Leith prior（純數據驅動）
  - ✅ Exponential LR Decay（與 RANS 版本相同）
  - ❌ 無 Curriculum（固定權重）

**其他可用配置**：
- `configs/templates/2d_quick_baseline.yml` - 快速基線模板（適合調試）
- `configs/templates/3d_production.yml` - 3D 生產級模板

### 🔬 實驗設計原則
根據專案 SOP（`CLAUDE.md`），進行對比實驗時：
1. **單一變因**：Leith prior vs Vanilla 只改變 prior 和 curriculum
2. **控制變量**：相同的 seed (42)、感測器、模型架構、學習率策略
3. **評估標準**：相對 L2 誤差、物理殘差、壓力梯度（詳見 `docs/EXPERIMENT_COMPARISON_PLAN.md`）

### 📝 文獻
- Musacchio & Boffetta (2014) - Kolmogorov Flow Reynolds 數定義
- Raissi et al. (2019) - Physics-Informed Neural Networks
- Wang et al. (2021) - VS-PINN 變數縮放

---

**專案倉庫**: https://github.com/latteine1217/pinns-mvp  
**授權**: MIT License